### LightGBM 

LightGBM was still on its default parameters. Tuning on val, with clean val alongside, same features as run #13. Early stopping on val picks the number of trees for each try, n_estimators 5000 is only a ceiling.

In [ ]:
from itertools import product

import pandas as pd

from prc_fuel.clean import clean_all, clean_training, mask_bad_values
from prc_fuel.evaluate import score
from prc_fuel.features import FEATURES
from prc_fuel.load import load
from prc_fuel.split import split
from prc_fuel.train import train

df = mask_bad_values(load("train"))
fit, val = split(df)
fit = clean_training(fit)
clean_val = clean_all(val)
features = [f for f in FEATURES if f not in ["avg_vertical_rate", "openap_fuel"]]

In [ ]:
params = {"learning_rate": 0.05, "n_estimators": 5000, "verbose": -1}
model = train(fit, features, params, val=val)

model.best_iteration_, score(model, val, features), score(model, clean_val, features)

learning_rate 0.05 with early stopping on val: stops at 233 trees, val 293.4, clean val 161.0 against 295.0 / 163.8 for #13 (defaults, 100 trees). Small gain on both.

In [ ]:
results = []
for num_leaves, min_child_samples in product([15, 31, 63, 127], [20, 50, 100]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results).sort_values("val")

More leaves is better on both val and clean val, 15 and 31 (default) are all at the bottom. min_child_samples is less clear: 20 is best on val with many leaves, 50-100 a bit better on clean val, gaps of 2-3 kg. Best on val 127 / 20: 288.1 / 157.0, about -7 kg on both against #13. 127 is the largest value tried and still the best, so the grid has to go higher.

In [ ]:
from itertools import product

results = []
for num_leaves, min_child_samples in product([127, 255, 511], [20, 50, 100]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results).sort_values("val")

Going above 127 leaves doesn't help on val (289.8-292.4 for 255 and 511), only 1-2 kg on clean val, noise level. Plateau reached. Kept num_leaves 127, min_child_samples 20: best on val, and the simpler of the near-ties on clean val.

In [ ]:
results = []
for learning_rate in [0.05, 0.02, 0.01]:
    params = {
        "learning_rate": learning_rate,
        "n_estimators": 5000,
        "num_leaves": 127,
        "min_child_samples": 20,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "learning_rate": learning_rate,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results)

A smaller learning rate doesn't help: 0.02 and 0.01 are worse on val (288.9, 289.8), 0.02 is ~1 kg better on clean val, noise level. Number of trees roughly doubles each time (160, 381, 731). Kept 0.05.

Retained: learning_rate 0.05, 160 trees, num_leaves 127, min_child_samples 20.

Run #15 with these as defaults (no early stopping, 160 trees): val 288.1, clean val 157.0, but rank 247.3 against 241.2 for #13. Better on val, worse on rank. val flights are drawn at random from April-August, so they have very close flights in fit (same aircraft, route, week), and bigger trees learn those details. rank is September, new flights, where those details don't carry over. Tuning on val picked the wrong thing, a time-based validation is needed before tuning again.

## Time-based validation

split_by_time puts the last 20% of flights to take off in val (from ~11 July), the rest in fit, like rank comes after train. Run #13 with it: val 188.0, clean val 159.5, rank 250.4. Few outliers in this val (188 vs 159.5 clean, against 295 vs 164 before), and rank is ~9 kg worse when the model hasn't seen mid-July to August, so recent data matters.

The test that counts: does this val react like rank? First on the config that fooled the random val.

In [ ]:
from prc_fuel.split import split_by_time

fit_t, val_t = split_by_time(df)
fit_t = clean_training(fit_t)
rank = mask_bad_values(load("rank"))

configs = {
    "default": {
        "learning_rate": 0.1,
        "n_estimators": 100,
        "num_leaves": 31,
        "min_child_samples": 20,
    },
    "tuned_on_random_val": {
        "learning_rate": 0.05,
        "n_estimators": 160,
        "num_leaves": 127,
        "min_child_samples": 20,
    },
}

pd.DataFrame(
    {
        name: {
            "val_time": score(m, val_t, features),
            "rank": score(m, rank, features),
        }
        for name, params in configs.items()
        for m in [train(fit_t, features, params)]
    }
)

The config tuned on random val is better on val_time (184.5 vs 188.0) but slightly worse on rank (251.1 vs 250.4). Much smaller disagreement than with the random split (rank +6 there), the big trees hurt mostly when fit and val share close flights. One pair isn't enough to judge, so comparing how both rank the 12 configs of the first grid.

In [ ]:
from itertools import product

results = []
for num_leaves, min_child_samples in product([15, 31, 63, 127], [20, 50, 100]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit_t, features, params, val=val_t)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val_time": score(model, val_t, features),
            "rank": score(model, rank, features),
        }
    )

grid_time = pd.DataFrame(results).sort_values("val_time")
grid_time

In [ ]:
grid_time[["val_time", "rank"]].corr(method="spearman")

Spearman 0.94 between val_time and rank over the 12 configs: the time-based val ranks the configs almost exactly like rank, so tuning can be done on it.

Same trends on both: more leaves is better (127 top, 15 bottom), and a higher min_child_samples (50-100) beats 20. The opposite of the random val, which favoured 20: bigger leaves can't learn the details of a few close flights, so they carry over better to later months. 10-11 kg between best and worst on both, well above noise. Best ones are on the edge of the grid (127, 100), extending it.

In [ ]:
results = []
for num_leaves, min_child_samples in product([127, 255], [100, 200, 400]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit_t, features, params, val=val_t)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val_time": score(model, val_t, features),
            "rank": score(model, rank, features),
        }
    )

grid_time2 = pd.DataFrame(results).sort_values("val_time")
grid_time2

min_child_samples 100 is the optimum: 200 loses ~1 kg and 400 ~5 kg, on both val_time and rank. 255 leaves is only 0.6-1 kg better than 127, noise level. Kept num_leaves 127, min_child_samples 100.

In [ ]:
results = []
for learning_rate in [0.05, 0.02, 0.01]:
    params = {
        "learning_rate": learning_rate,
        "n_estimators": 5000,
        "num_leaves": 127,
        "min_child_samples": 100,
        "verbose": -1,
    }
    model = train(fit_t, features, params, val=val_t)
    results.append(
        {
            "learning_rate": learning_rate,
            "n_trees": model.best_iteration_,
            "val_time": score(model, val_t, features),
            "rank": score(model, rank, features),
        }
    )

pd.DataFrame(results)

A smaller learning rate changes nothing: -0.4 to -0.5 kg on val_time, +0.6 to +0.7 on rank, noise in opposite directions. Kept 0.05, same quality with 4-5 times fewer trees.

Retained: learning_rate 0.05, 75 trees (early stopping), num_leaves 127, min_child_samples 100. With the time split, against the defaults on the same fit: val_time 175.8 vs 188.0, clean val_time 146.6 vs 159.5, rank 243.5 vs 250.4.

With the random split rank doesn't move (241.9 vs 241.2 for #13) and val is a bit worse (298.0 vs 295.0) while clean val is better (156.8 vs 163.8): leaves of at least 100 intervals smooth out the outlier flights, which weigh a lot in that val. No clear reason why the rank gain disappears there, the random fit has August flights the time fit doesn't.

### CatBoost

CatBoost with its defaults (run #20) gets 187.1 on val_time but 238.1 on rank, best rank so far, while tuned LightGBM is 175.8 / 243.5. val and rank disagree, so tuning it on the time split before comparing. Same method as for LightGBM: learning rate 0.05, early stopping on val_time, 5000 trees only a ceiling.
depth is the size of its trees (symmetric trees, so no num_leaves), l2_leaf_reg a penalty that keeps leaves from fitting the training flights too closely.

In [ ]:
results = []
for depth, l2_leaf_reg in product([4, 6, 8, 10], [3, 10, 30]):
    params = {
        "learning_rate": 0.05,
        "iterations": 5000,
        "depth": depth,
        "l2_leaf_reg": l2_leaf_reg,
    }
    model = train(fit_t, features, params, val=val_t, algo="catboost")
    results.append(
        {
            "depth": depth,
            "l2_leaf_reg": l2_leaf_reg,
            "n_trees": model.model.tree_count_,
            "val_time": score(model, val_t, features),
            "rank": score(model, rank, features),
        }
    )

grid_cat = pd.DataFrame(results).sort_values("val_time")
grid_cat

In [ ]:
grid_cat[["val_time", "rank"]].corr(method="spearman")

Spearman val_time vs rank only 0.59 here, against 0.94 for LightGBM. The top configs are within 1-3 kg, noise level, and l2_leaf_reg 30 hurts rank with deep trees (~249-250) without showing on val.
Depth matters most: 4 is the worst (~202), 8-10 the best (~180).
Best on val_time: depth 10, l2_leaf_reg 3, 226 trees, 179.6 / rank 241.3.
Still ~4 kg behind tuned LightGBM on val_time (175.8) but ~2 kg better on rank (243.5). Every config is behind LightGBM on val_time, most good ones are level or better on rank. Same disagreement as run #20, it doesn't come from bad defaults.
LightGBM stays the reference (decisions on val_time). The two models seem to make different errors, so averaging them is worth a try.

### Averaging LightGBM and CatBoost

Both trained on the same fit with their best settings: LightGBM with DEFAULT_PARAMS (#17), CatBoost depth 10, l2_leaf_reg 3, 226 trees at 0.05 (fixed, no early stopping, so val_time isn't used to train it). Prediction = (1 - w) * LightGBM + w * CatBoost, w picked on val_time.

In [ ]:
from prc_fuel.evaluate import rmse
from prc_fuel.features import build_features

lgb = train(fit_t, features)
cat = train(
    fit_t,
    features,
    {"learning_rate": 0.05, "iterations": 226, "depth": 10, "l2_leaf_reg": 3},
    algo="catboost",
)

X_val = build_features(val_t, features)
X_rank = build_features(rank, features)
pred = {
    "val_time": (lgb.predict(X_val), cat.predict(X_val)),
    "rank": (lgb.predict(X_rank), cat.predict(X_rank)),
}
truth = {"val_time": val_t["fuel_kg"], "rank": rank["fuel_kg"]}

results = []
for w in [0, 0.25, 0.5, 0.75, 1]:
    row = {"catboost_weight": w}
    for name, (p_lgb, p_cat) in pred.items():
        row[name] = rmse(truth[name], (1 - w) * p_lgb + w * p_cat)
    results.append(row)

pd.DataFrame(results)

Both ends match the single models (175.8 and 179.6), so the average is computed right.
w = 0.5 is the best on val_time: 170.0, -5.8 kg against LightGBM alone. Rank 238.6, -5 kg, close to the best rank so far (238.1, CatBoost defaults).
This time val_time and rank agree, both get clearly better. U shape: each model alone is worse than the mix, they probably make different errors and the average cancels part of them.
0.5 vs 0.75 on rank is 0.4 kg, noise. Kept 0.5, best on val_time and the simplest.

### CatBoost on the fuel rate

Learning fuel_kg / duration_s and multiplying back by the duration (run #25) made CatBoost the best model: 162.6 on val_time, 232.9 on rank. Its settings were tuned for the fuel in kg, so tuning again for the rate. depth goes up to 12 and l2_leaf_reg down to 1, since the best values were on the edge of the first grid (10 and 3).

In [ ]:
from itertools import product

results = []
for depth, l2_leaf_reg in product([6, 8, 10, 12], [1, 3, 10]):
    params = {
        "learning_rate": 0.05,
        "iterations": 5000,
        "depth": depth,
        "l2_leaf_reg": l2_leaf_reg,
    }
    model = train(fit_t, features, params, val=val_t, algo="catboost", target="rate")
    results.append(
        {
            "depth": depth,
            "l2_leaf_reg": l2_leaf_reg,
            "n_trees": model.model.model.tree_count_,
            "val_time": score(model, val_t, features),
            "rank": score(model, rank, features),
        }
    )

grid_rate = pd.DataFrame(results).sort_values("val_time")
grid_rate

No config beats the current settings (depth 10, l2_leaf_reg 3, 226 fixed trees: 162.6 on val_time). Best here depth 12 / l2_leaf_reg 10 with 164.0.
Same settings as now (depth 10, l2_leaf_reg 3) give 166.0 here, early stopping cut them at 160 trees. It watches the error on the rate, where every interval weighs the same, while the score is in kg where long intervals weigh much more, so it probably stops too early.
Top configs within 1-2 kg, noise level. Kept the current settings.

In [ ]:
grid_rate[["val_time", "rank"]].corr(method="spearman")

Spearman 0.52 between val_time and rank, against 0.94 for LightGBM on kg. The grid is probably biased by early stopping (if every config stops a bit too early, not at the same point), so its order can't be trusted much. A fair grid would need fixed numbers of trees, small gain expected given the gaps.